# Notebook 4 — Evaluation, Statistical Testing & Scalability

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Objectives
1. Evaluate all **4 models** on the **held-out TEST set** (unbiased estimate).
2. Compute: Accuracy, F1, Precision, Recall, ROC-AUC.
3. Plot confusion matrices with **explicit TP / FP / FN / TN**.
4. Extract & visualise **top 20 feature importance** words (RF + MurmurHash3 reverse lookup).
5. Generate ROC curves.
6. **Statistical Significance Testing** — bootstrap 95% CI + pairwise McNemar tests.
7. **scikit-learn baseline comparison** — distributed (Spark) vs single-node (4 models).
8. **Scalability experiments** — weak scaling (data size) + strong scaling (parallelism).
9. Export all results for Tableau dashboards.

In [ ]:
# ── 4.1  Bootstrap ──────────────────────────────────────────────────────
import os, sys
os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations
from scipy import stats as scipy_stats

from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from pyspark import StorageLevel
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)

spark = (
    SparkSession.builder
    .appName("FakeNewsDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

TABLEAU_DIR = Path("../tableau")
TABLEAU_DIR.mkdir(exist_ok=True)

In [ ]:
# ── 4.2  Load test set & saved models (4 algorithms) ────────────────────
from pyspark.ml.classification import (
    LogisticRegressionModel, LinearSVCModel,
    RandomForestClassificationModel, NaiveBayesModel
)

test_df = spark.read.parquet("../data/parquet/features/test").persist(StorageLevel.MEMORY_AND_DISK)
print(f"Test set: {test_df.count():,} rows")

MODELS_DIR = Path("../data/models")

models = {
    "LogisticRegression": LogisticRegressionModel.load(str(MODELS_DIR / "LogisticRegression_mllib")),
    "LinearSVC":          LinearSVCModel.load(str(MODELS_DIR / "LinearSVC_mllib")),
    "RandomForest":       RandomForestClassificationModel.load(str(MODELS_DIR / "RandomForest_mllib")),
    "NaiveBayes":         NaiveBayesModel.load(str(MODELS_DIR / "NaiveBayes_mllib")),
}

print(f"Loaded {len(models)} models")

In [ ]:
# ── 4.3  Full evaluation on TEST set ────────────────────────────────────

auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
precision_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)

test_results = []
predictions_dict = {}   # store for confusion matrix

for name, model in models.items():
    preds = model.transform(test_df)
    predictions_dict[name] = preds
    
    accuracy  = acc_eval.evaluate(preds)
    f1        = f1_eval.evaluate(preds)
    precision = precision_eval.evaluate(preds)
    recall    = recall_eval.evaluate(preds)
    
    try:
        auc = auc_eval.evaluate(preds)
    except Exception:
        auc = None
    
    test_results.append({
        "model":     name,
        "accuracy":  round(accuracy, 4),
        "f1":        round(f1, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "auc":       round(auc, 4) if auc else None,
    })

results_df = pd.DataFrame(test_results)
print(results_df.to_string(index=False))

In [ ]:
# ── 4.4  Confusion Matrices with TP / FP / FN / TN (4 models) ───────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

n_models = len(predictions_dict)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]
cm_data = []

for ax, (name, preds) in zip(axes, predictions_dict.items()):
    pdf = preds.select("label", "prediction").toPandas()
    cm = confusion_matrix(pdf["label"], pdf["prediction"], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()    # EXPLICIT TP/FP/FN/TN extraction
    cm_data.append({"model": name, "TP": tp, "FP": fp, "FN": fn, "TN": tn})
    print(f"{name}: TP={tp}  FP={fp}  FN={fn}  TN={tn}")

    disp = ConfusionMatrixDisplay(cm, display_labels=["Reliable", "Fake"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    acc_val = results_df.loc[results_df["model"]==name, "accuracy"].values[0]
    ax.set_title(f"{name}\nAcc={acc_val}")

plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame(cm_data).to_csv(str(TABLEAU_DIR / "confusion_matrix_details.csv"), index=False)
print("\n✓ Saved confusion_matrices.png + confusion_matrix_details.csv")

In [ ]:
# ── 4.5  ROC Curve Data — PySpark 4.x compatible ────────────────────────
from sklearn.metrics import roc_curve, auc

roc_data_all = []
fig, ax = plt.subplots(figsize=(8, 6))

for name, preds in predictions_dict.items():
    # PySpark 4.x: probability column is Vector UDT — must use vector_to_array()
    if "probability" in preds.columns:
        pdf = preds.select(
            "label", vector_to_array("probability").getItem(1).alias("prob_1")
        ).toPandas()
    else:
        # LinearSVC: no probability column, use rawPrediction
        pdf = preds.select(
            "label", vector_to_array("rawPrediction").getItem(1).alias("prob_1")
        ).toPandas()
    
    fpr, tpr, _ = roc_curve(pdf["label"], pdf["prob_1"])
    roc_auc = auc(fpr, tpr)
    
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")
    
    for f, t in zip(fpr, tpr):
        roc_data_all.append({"model": name, "fpr": f, "tpr": t, "auc": roc_auc})

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Test Set")
ax.legend()
plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "roc_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

roc_df = pd.DataFrame(roc_data_all)
roc_df.to_csv(str(TABLEAU_DIR / "roc_data.csv"), index=False)
print(f"✓ Exported {len(roc_df):,} ROC data points")

In [ ]:
# ── 4.6  Feature Importance (Random Forest) — Top 20 ────────────────────
rf_model = models["RandomForest"]
importances = rf_model.featureImportances.toArray()

top_k = 20
top_indices = np.argsort(importances)[::-1][:top_k]
top_scores  = importances[top_indices]

fi_df = pd.DataFrame({
    "feature_index": top_indices,
    "importance": top_scores,
    "rank": range(1, top_k + 1)
})
fi_df["feature_label"] = fi_df["feature_index"].apply(lambda x: f"hash_{x}")
print(f"Top {top_k} features extracted from RF ({len(importances):,}-dim vector)")

In [ ]:
# ── 4.7  Reverse hash lookup — map hash indices to actual words ──────────
import mmh3, re
from collections import defaultdict
import pyarrow.parquet as pq

NUM_FEATURES = 2**16   # Must match NB2's HashingTF numFeatures
top_set = set(int(i) for i in top_indices)

# Map for the 5 custom text-statistics features (appended after TF-IDF by VectorAssembler)
TEXT_STATS_NAMES = {
    NUM_FEATURES:     "text_length",
    NUM_FEATURES + 1: "word_count",
    NUM_FEATURES + 2: "avg_word_length",
    NUM_FEATURES + 3: "caps_ratio",
    NUM_FEATURES + 4: "exclamation_count",
}

# Read raw texts with PyArrow (avoids Spark Python worker issues on Windows)
articles_path = "../data/parquet/news_articles"
texts = pq.read_table(articles_path, columns=["text"]).column("text").to_pylist()

STOP_WORDS = set("i me my myself we our ours ourselves you your yours yourself yourselves "
    "he him his himself she her hers herself it its itself they them their theirs "
    "themselves what which who whom this that these those am is are was were be been "
    "being have has had having do does did doing a an the and but if or because as "
    "until while of at by for with about against between through during before after "
    "above below to from up down in out on off over under again further then once "
    "here there when where why how all both each few more most other some such no "
    "nor not only own same so than too very s t can will just don should now".split())

unique_words = set()
for text in texts:
    if text:
        for w in re.findall(r'\b[a-z]{2,}\b', text.lower()):
            if w not in STOP_WORDS:
                unique_words.add(w)

# Map words → Spark hash buckets using MurmurHash3 (seed=42)
hash_to_words = defaultdict(list)
for w in unique_words:
    h = mmh3.hash(w, seed=42, signed=True)
    bucket = ((h % NUM_FEATURES) + NUM_FEATURES) % NUM_FEATURES
    if bucket in top_set:
        hash_to_words[bucket].append(w)

# Look up: first check text_stats names, then hash lookup
fi_df["words"] = fi_df["feature_index"].apply(
    lambda x: TEXT_STATS_NAMES.get(int(x),
              ", ".join(sorted(hash_to_words.get(int(x), ["--"]))[:3]))
)
fi_df["feature_label"] = fi_df.apply(
    lambda r: r["words"][:30] if r["words"] != "--" else f"hash_{r['feature_index']}", axis=1
)

# Plot with word labels
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=fi_df, x="importance", y="feature_label", hue="feature_label",
            ax=ax, palette="viridis", legend=False)
ax.set_title("Top 20 Predictive Features (Random Forest Feature Importance)")
ax.set_xlabel("Gini Importance"); ax.set_ylabel("")
plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "feature_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

fi_df.to_csv(str(TABLEAU_DIR / "feature_importance.csv"), index=False)
fi_df.to_csv(str(TABLEAU_DIR / "feature_importance_with_words.csv"), index=False)
print("\n=== TOP 20 PREDICTIVE FEATURES ===")
print(fi_df[["rank", "importance", "words"]].to_string(index=False))

## 4.8  Statistical Significance Testing

Two complementary tests ensure our model comparisons are **statistically rigorous**:

1. **Bootstrap 95% Confidence Intervals (n=1000)** — non-parametric estimate of the uncertainty around each model's F1 and Accuracy scores.  Narrow, non-overlapping CIs indicate a reliable performance difference.

2. **McNemar Test (pairwise)** — compares *which individual samples* two models classify differently.  Unlike comparing aggregate metrics, McNemar's test detects whether the *pattern* of errors is statistically significant (chi-squared with continuity correction, α = 0.05).

In [ ]:
# ── 4.8a  Bootstrap 95% Confidence Intervals ─────────────────────────────
from sklearn.metrics import accuracy_score, f1_score

# Collect all predictions to pandas (once, for efficiency)
pred_pandas = {}
for name, preds in predictions_dict.items():
    pred_pandas[name] = preds.select("label", "prediction").toPandas()

N_BOOTSTRAP = 1000
np.random.seed(42)
bootstrap_results = []

print("=== BOOTSTRAP 95% CONFIDENCE INTERVALS (n=1000) ===")
for name, pdf in pred_pandas.items():
    y_true = pdf["label"].values
    y_pred = pdf["prediction"].values
    n = len(y_true)

    boot_f1s, boot_accs = [], []
    for _ in range(N_BOOTSTRAP):
        idx = np.random.choice(n, n, replace=True)
        boot_f1s.append(f1_score(y_true[idx], y_pred[idx]))
        boot_accs.append(accuracy_score(y_true[idx], y_pred[idx]))

    ci_f1_lo, ci_f1_hi   = np.percentile(boot_f1s, [2.5, 97.5])
    ci_acc_lo, ci_acc_hi = np.percentile(boot_accs, [2.5, 97.5])

    bootstrap_results.append({
        "model": name,
        "f1_mean": round(np.mean(boot_f1s), 4),
        "f1_ci_lower": round(ci_f1_lo, 4),
        "f1_ci_upper": round(ci_f1_hi, 4),
        "acc_mean": round(np.mean(boot_accs), 4),
        "acc_ci_lower": round(ci_acc_lo, 4),
        "acc_ci_upper": round(ci_acc_hi, 4),
    })
    print(f"  {name:25s}  F1={np.mean(boot_f1s):.4f} [{ci_f1_lo:.4f}, {ci_f1_hi:.4f}]"
          f"  Acc={np.mean(boot_accs):.4f} [{ci_acc_lo:.4f}, {ci_acc_hi:.4f}]")

boot_df = pd.DataFrame(bootstrap_results)
boot_df.to_csv(str(TABLEAU_DIR / "bootstrap_confidence_intervals.csv"), index=False)
print("\n✓ Exported bootstrap_confidence_intervals.csv")

# ── Bootstrap CI bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(boot_df))
ax.barh(x_pos, boot_df["f1_mean"],
        xerr=[boot_df["f1_mean"] - boot_df["f1_ci_lower"],
              boot_df["f1_ci_upper"] - boot_df["f1_mean"]],
        capsize=5, color="steelblue", alpha=0.8)
ax.set_yticks(x_pos)
ax.set_yticklabels(boot_df["model"])
ax.set_xlabel("F1 Score")
ax.set_title("Bootstrap 95% Confidence Intervals — F1 (n=1000)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "bootstrap_ci_chart.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 4.8b  McNemar Test (pairwise model comparison) ───────────────────────
print("=== McNEMAR TEST (pairwise, α=0.05 with continuity correction) ===")
model_names = list(pred_pandas.keys())
mcnemar_results = []

for m1, m2 in combinations(model_names, 2):
    y_true = pred_pandas[m1]["label"].values
    y1     = pred_pandas[m1]["prediction"].values
    y2     = pred_pandas[m2]["prediction"].values

    # McNemar contingency: b = model1 correct & model2 wrong, c = vice versa
    b = int(np.sum((y1 == y_true) & (y2 != y_true)))
    c = int(np.sum((y1 != y_true) & (y2 == y_true)))

    if b + c == 0:
        chi2_stat, p_value = 0.0, 1.0
    else:
        chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
        p_value   = 1 - scipy_stats.chi2.cdf(chi2_stat, df=1)

    sig = "YES ✗" if p_value < 0.05 else "no"
    mcnemar_results.append({
        "model_1": m1, "model_2": m2,
        "b_correct1_wrong2": b, "c_wrong1_correct2": c,
        "chi2": round(chi2_stat, 4), "p_value": round(p_value, 6),
        "significant_005": p_value < 0.05,
    })
    print(f"  {m1:25s} vs {m2:25s}  chi2={chi2_stat:8.4f}  p={p_value:.6f}  sig={sig}")

mcnemar_df = pd.DataFrame(mcnemar_results)
mcnemar_df.to_csv(str(TABLEAU_DIR / "mcnemar_tests.csv"), index=False)
print("\n✓ Exported mcnemar_tests.csv")

In [ ]:
# ── 4.9  Scikit-learn Baseline Comparison (4 models) ─────────────────────
import time, pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.svm import LinearSVC as SklearnSVC
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.naive_bayes import MultinomialNB as SklearnNB
from sklearn.model_selection import train_test_split

# Load raw text data
all_pdf = pq.read_table("../data/parquet/news_articles", columns=["text", "label"]).to_pandas()

X_train_sk, X_test_sk, y_train_sk, y_test_sk = train_test_split(
    all_pdf["text"].values, all_pdf["label"].values,
    test_size=0.3, random_state=42, stratify=all_pdf["label"].values
)
print(f"sklearn Train: {len(X_train_sk):,}  |  Test: {len(X_test_sk):,}")

# TF-IDF (single-node equivalent of Spark HashingTF→IDF pipeline)
t0 = time.time()
tfidf = TfidfVectorizer(max_features=NUM_FEATURES, stop_words="english", min_df=5)
X_train_tfidf = tfidf.fit_transform(X_train_sk)
X_test_tfidf  = tfidf.transform(X_test_sk)
vec_time = time.time() - t0
print(f"TF-IDF vectorization: {vec_time:.2f}s")

sklearn_results = []
sklearn_models = {
    "sklearn_LR":  SklearnLR(max_iter=100, C=10.0, solver="lbfgs", random_state=42),
    "sklearn_SVC": SklearnSVC(max_iter=100, C=10.0, random_state=42),
    "sklearn_RF":  SklearnRF(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1),
    "sklearn_NB":  SklearnNB(alpha=1.0),
}
for sk_name, sk_model in sklearn_models.items():
    t0 = time.time()
    sk_model.fit(X_train_tfidf, y_train_sk)
    sk_time = time.time() - t0
    y_pred = sk_model.predict(X_test_tfidf)
    sk_acc = accuracy_score(y_test_sk, y_pred)
    sk_f1  = f1_score(y_test_sk, y_pred)
    sklearn_results.append({"model": sk_name, "accuracy": round(sk_acc, 4),
                            "f1": round(sk_f1, 4), "train_time_s": round(sk_time, 2)})
    print(f"  {sk_name:15s}  Acc={sk_acc:.4f}  F1={sk_f1:.4f}  Time={sk_time:.2f}s")

sklearn_df = pd.DataFrame(sklearn_results)
sklearn_df.to_csv(str(TABLEAU_DIR / "sklearn_baseline.csv"), index=False)

# ── Pickle sklearn models (serialization) ──
print("\n=== SKLEARN MODEL SERIALIZATION (Pickle) ===")
for sk_name, sk_model in sklearn_models.items():
    pkl_path = str(MODELS_DIR / f"{sk_name}.pkl")
    with open(pkl_path, "wb") as fh:
        pickle.dump(sk_model, fh)
    with open(pkl_path, "rb") as fh:
        loaded_sk = pickle.load(fh)
    pkl_size = Path(pkl_path).stat().st_size / 1024
    print(f"  {sk_name}: {pkl_path} ({pkl_size:.1f} KB)  Verified: {type(loaded_sk).__name__}")

print("\n✓ Exported sklearn_baseline.csv + pickled models")

In [ ]:
# ── 4.10  Distributed (Spark) vs Single-Node (scikit-learn) Comparison ──
comparison_rows = []
spark_names = ["LogisticRegression", "LinearSVC", "RandomForest", "NaiveBayes"]
sk_names    = ["sklearn_LR", "sklearn_SVC", "sklearn_RF", "sklearn_NB"]

# Load Spark validation metrics (from model_comparison.csv)
val_metrics_df = pd.read_csv(str(TABLEAU_DIR / "model_comparison.csv"))

for sp_name, sk_name in zip(spark_names, sk_names):
    sp_row = results_df[results_df["model"] == sp_name].iloc[0]
    sk_row = sklearn_df[sklearn_df["model"] == sk_name].iloc[0]
    comparison_rows.append({
        "Algorithm": sp_name,
        "Spark_Acc": sp_row["accuracy"],
        "Spark_F1": sp_row["f1"],
        "Spark_Time_s": val_metrics_df[val_metrics_df["model"]==sp_name]["train_time_s"].values[0],
        "sklearn_Acc": sk_row["accuracy"],
        "sklearn_F1": sk_row["f1"],
        "sklearn_Time_s": sk_row["train_time_s"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(str(TABLEAU_DIR / "distributed_vs_singlenode.csv"), index=False)
print("=== DISTRIBUTED (Spark MLlib) vs SINGLE-NODE (scikit-learn) ===")
print(comparison_df.to_string(index=False))
print("\n✓ Exported distributed_vs_singlenode.csv")

In [ ]:
# ── 4.10  Scalability Experiments (Weak + Strong Scaling) ────────────────
from pyspark.ml.classification import LogisticRegression as SparkLR

# ── Weak Scaling: vary data size, fixed resources ──
print("--- Weak Scaling (data size) ---")
weak_scaling = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    sample = spark.read.parquet("../data/parquet/features/train").sample(fraction=frac, seed=42)
    sample.persist(StorageLevel.MEMORY_AND_DISK)
    n = sample.count()
    lr_s = SparkLR(featuresCol="features", labelCol="label", maxIter=20)
    t0 = time.time()
    lr_s.fit(sample)
    elapsed = time.time() - t0
    weak_scaling.append({"data_fraction": frac, "num_rows": n, "train_time_s": round(elapsed, 2)})
    print(f"  frac={frac:.2f}  rows={n:>8,}  time={elapsed:.2f}s")
    sample.unpersist()

weak_df = pd.DataFrame(weak_scaling)
weak_df.to_csv(str(TABLEAU_DIR / "weak_scaling.csv"), index=False)

# ── Strong Scaling: vary parallelism (core count), fixed data ──
print("\n--- Strong Scaling (parallelism) ---")
strong_scaling = []
full_train = spark.read.parquet("../data/parquet/features/train")
full_train.persist(StorageLevel.MEMORY_AND_DISK)
full_train.count()

for n_cores in [1, 2, 4]:
    repartitioned = full_train.repartition(n_cores)
    repartitioned.persist(StorageLevel.MEMORY_AND_DISK)
    repartitioned.count()
    lr_s = SparkLR(featuresCol="features", labelCol="label", maxIter=20)
    t0 = time.time()
    lr_s.fit(repartitioned)
    elapsed = time.time() - t0
    strong_scaling.append({"num_cores": n_cores, "num_partitions": n_cores, "train_time_s": round(elapsed, 2)})
    print(f"  cores={n_cores}  partitions={n_cores}  time={elapsed:.2f}s")
    repartitioned.unpersist()

full_train.unpersist()
strong_df = pd.DataFrame(strong_scaling)
strong_df.to_csv(str(TABLEAU_DIR / "strong_scaling.csv"), index=False)

# ── Combined CSV for Tableau ──
scaling_combined = []
for _, r in weak_df.iterrows():
    scaling_combined.append({"experiment": "weak_scaling", "variable": f"{r['data_fraction']:.0%} data",
                             "value": r["num_rows"], "train_time_s": r["train_time_s"]})
for _, r in strong_df.iterrows():
    scaling_combined.append({"experiment": "strong_scaling", "variable": f"{r['num_cores']} cores",
                             "value": r["num_cores"], "train_time_s": r["train_time_s"]})
pd.DataFrame(scaling_combined).to_csv(str(TABLEAU_DIR / "scaling_experiments.csv"), index=False)
print("\n✓ Exported scaling CSVs")

In [ ]:
# ── 4.11  Scaling Plots + Speedup Analysis ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Weak scaling plot
ax1.plot(weak_df["num_rows"], weak_df["train_time_s"], "o-", linewidth=2, markersize=8, color="steelblue")
ax1.set_xlabel("Number of Training Rows", fontsize=11)
ax1.set_ylabel("Training Time (s)", fontsize=11)
ax1.set_title("Weak Scaling: LR Training Time vs Data Size", fontsize=12)
ax1.grid(True, alpha=0.3)
for _, r in weak_df.iterrows():
    ax1.annotate(f"{r['train_time_s']:.1f}s", (r["num_rows"], r["train_time_s"]),
                 textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)

# Strong scaling plot
ax2.plot(strong_df["num_cores"], strong_df["train_time_s"], "s-", linewidth=2, markersize=8, color="coral")
ax2.set_xlabel("Number of Cores (Partitions)", fontsize=11)
ax2.set_ylabel("Training Time (s)", fontsize=11)
ax2.set_title("Strong Scaling: LR Training Time vs Parallelism", fontsize=12)
ax2.set_xticks([1, 2, 4])
ax2.grid(True, alpha=0.3)
for _, r in strong_df.iterrows():
    ax2.annotate(f"{r['train_time_s']:.1f}s", (r["num_cores"], r["train_time_s"]),
                 textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "scaling_plot.png"), dpi=150, bbox_inches="tight")
plt.show()

# Speedup analysis
baseline_time = strong_df.loc[strong_df["num_cores"]==1, "train_time_s"].values[0]
print("\n=== SPEEDUP ANALYSIS (Strong Scaling) ===")
for _, r in strong_df.iterrows():
    speedup = baseline_time / r["train_time_s"]
    efficiency = speedup / r["num_cores"] * 100
    print(f"  {int(r['num_cores'])} cores: {r['train_time_s']:.2f}s  |  Speedup: {speedup:.2f}x  |  Efficiency: {efficiency:.1f}%")

In [ ]:
# ── 4.12  Final Export Summary + Cleanup ─────────────────────────────────
results_df.to_csv(str(TABLEAU_DIR / "test_metrics.csv"), index=False)

print("=== TABLEAU EXPORT INVENTORY ===")
for f in sorted(TABLEAU_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s} {size:>8.1f} KB")

test_df.unpersist()
print("\n✓ Notebook 4 complete — all evaluations, comparisons, and exports done.")